# Notebook BI Platform

Opérations communes sur une plateforme SAP BI 4.3
- connexion / déconnexion
- récupération des informations standards ou détaillées d'un document WebI
- récupération des informations standards ou détaillées des fournisseurs de données

In [1]:
APPLICATION_NAME = "BIP43_Python"

In [ ]:
import sys
from pathlib import Path

# Remonter les dossiers jusqu'à trouver lib/init/init_code.py
def find_app_home(sentinel: str ="lib/bootstrap/bootstrap.py"):
    current = Path.cwd().resolve()
    root = current.root
    while current != root:
        if (current / sentinel).is_file():
            return current
        current = current.parent
    raise FileNotFoundError(f"Impossible de trouver le fichier sentinelle : {sentinel}")

# Trouver et ajouter APPLICATION_HOME au sys.path
APPLICATION_HOME = find_app_home()
sys.path.insert(0, str(APPLICATION_HOME))
#print(f"APPLICATION_HOME set to: {APPLICATION_HOME}")

from lib.bootstrap.bootstrap import init_env
epy = init_env()

## Chargement des classes et variables

In [3]:
props = epy.cfgprops
yml = epy.cfgyaml
log = epy.log

In [4]:
# information d'identification BIP 4.3
url = props.get("bo_url")
account = props.get("bo_account")
password = props.get("bo_password")
type_auth = props.get("bo_authentication")

In [5]:
bip = epy.load_class(module_name='bip', args=[APPLICATION_HOME, APPLICATION_NAME, url])

In [6]:
webi = epy.load_class(module_name='webi', args=[APPLICATION_HOME, APPLICATION_NAME, bip])

## Authentification & Accès

In [ ]:
# Connexion
token = bip.set_token(base_url=url, username=account, password=password, auth_type=type_auth)
print(token)

In [ ]:
# BASE_URL pour l'accès API
print(bip.get_bip_url)

# Affichage du token
print(bip.get_token)

## Liste des documents
- la liste des documents peut être enregistré dans 'APPLICATION_HOME/tmp/documents_list.txt'

In [9]:
# récupération du patrimoine (sans sauvegarde en fichier)
documents = webi.request_cms(props.get("list_all_documents"))

In [10]:
dossiers = webi.request_cms(props.get("list_all_folders"))

In [ ]:
# les FavoritesFolder sont les racines personnelles — pas besoin de SI_PATH
favorites_ids = {f["SI_ID"] for f in dossiers if f["SI_KIND"] == "FavoritesFolder"}

personal_folder_ids = webi.get_all_personal_folder_ids(dossiers, favorites_ids)
public_folder_ids   = {f["SI_ID"] for f in dossiers if f["SI_ID"] not in personal_folder_ids}

docs_in_private_folder = [d for d in documents if d["SI_PARENT_FOLDER"] in personal_folder_ids]
docs_in_public_folder  = [d for d in documents if d["SI_PARENT_FOLDER"] not in personal_folder_ids]

log.info(f"-- nombre de documents total: {len(documents)}")
log.info(f"-- nombre de documents 'privés': {len(docs_in_private_folder)}")
log.info(f"-- nombre de documents 'publics': {len(docs_in_public_folder)}")

log.log("")

In [ ]:
for d in documents:
    print(d)

## Information document

document id:
- dev: 15295
- qual: 
- preprod: 
- prod: 

In [13]:
doc_id: int = 155624

In [ ]:
# Affichage des informations générales du document
info = webi.get_doc_info(doc_id)
for inf in info.items():
    print(inf)

# Affichage d'un détail d'information générale du document
print(info.get('description'))

In [ ]:
# Récupération des invites utilisateurs pour actualisation (avec sauvegarde du fichier de paramétrage)
prompts = webi.get_doc_prompts(doc_id, saved=False)
print(prompts)

## Fournisseurs de données

In [ ]:
# Récupération des informations générales des fournisseurs de données
gen_dp = webi.get_doc_dp(doc_id)
print(type(gen_dp))
for dp in gen_dp:
    print(dp)

In [ ]:
detailled_dp = webi.get_dp_details(doc_id, simplified=True)
print(type(detailled_dp))
for dpd in detailled_dp:
    print(f"{dpd['name']} - id: {dpd['id']}")

## Deconnexion

In [18]:
# Deconnexion
bip.unset_token()